COM7021 Data Visualisation - EcoMove Mobility investigation
Data audit, preparation, summary tables and seven static visualisations.

In [1]:

import io, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

DATA = "EcoMoveMobility.xlsx"   # [5442] EcoMoveMobility dataset supplied with the brief
OUT = "figs/"
import os; os.makedirs(OUT, exist_ok=True)

# ---------------------------------------------------------------- style
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10.5,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.labelsize": 10.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e6e6", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "figure.dpi": 110, "savefig.dpi": 200,
    "savefig.bbox": "tight",
})
INK, MUTED, ACCENT, ALERT = "#1f2933", "#7b8794", "#0b6e99", "#c0392b"
# Okabe-Ito colour-blind-safe palette, fixed per transport mode for consistency
MODE_COL = {"Metro": "#0072B2", "Tram": "#009E73", "Bus": "#D55E00",
            "Park and Ride": "#CC79A7", "Bike Share": "#E69F00", "E-Scooter": "#56B4E9"}
COUNTRY_COL = {"Germany": "#0072B2", "France": "#009E73", "Spain": "#E69F00",
               "Italy": "#CC79A7", "Netherlands": "#D55E00", "Austria": "#56B4E9"}

def save(fig, name):
    fig.savefig(OUT + name, facecolor="white"); plt.close(fig)

# ---------------------------------------------------------------- 1. load & audit
df = pd.read_excel(DATA, sheet_name="Dataset")
buf = io.StringIO()
buf.write(f">>> df.shape\n{df.shape}\n\n>>> df.dtypes\n{df.dtypes.to_string()}\n\n")
buf.write(f">>> df.isna().sum().sum()\n{int(df.isna().sum().sum())}\n\n")
buf.write(f">>> df.duplicated().sum(), df['Record_ID'].duplicated().sum()\n"
          f"({int(df.duplicated().sum())}, {int(df['Record_ID'].duplicated().sum())})\n\n")
buf.write(">>> df['Date'].agg(['min','max'])  # Excel serial numbers\n"
          f"min {df['Date'].min()}   max {df['Date'].max()}\n")
audit_text = buf.getvalue()

# ---------------------------------------------------------------- 2. prepare
df["Date"] = pd.to_datetime(df["Date"], unit="D", origin="1899-12-30")
df["Month"] = df["Date"].dt.to_period("M").dt.to_timestamp()
df["Year"] = df["Date"].dt.year
df["Event_Day"] = df["Event_Day"].astype("category")
df["Net_Result_EUR"] = df["Ticket_Revenue_EUR"] - df["Operating_Cost_EUR"]
df["Revenue_per_Passenger"] = df["Ticket_Revenue_EUR"] / df["Passengers"]
df["Complaints_per_1k"] = df["Accessibility_Complaints"] / df["Passengers"] * 1000

# validity checks
checks = {
    "on_time_in_0_100": bool(df["On_Time_Percentage"].between(0, 100).all()),
    "rating_in_1_5": bool(df["Customer_Rating"].between(1, 5).all()),
    "negative_values": int((df.select_dtypes("number").drop(columns=["Net_Result_EUR"]) < 0).sum().sum()),
    "zero_energy_rows": int((df["Energy_Consumption_KWh"] == 0).sum()),
    "zero_energy_modes": df.loc[df["Energy_Consumption_KWh"] == 0, "Transport_Mode"].unique().tolist(),
    "records_per_day_mean": round(df.groupby("Date").size().mean(), 2),
}
audit_text += (f"\n>>> validity checks\nOn_Time_Percentage within 0-100: {checks['on_time_in_0_100']}\n"
               f"Customer_Rating within 1-5:      {checks['rating_in_1_5']}\n"
               f"Zero energy rows: {checks['zero_energy_rows']} (modes: {checks['zero_energy_modes']})\n"
               f"Mean records per calendar day: {checks['records_per_day_mean']}\n")
with open(OUT + "audit.txt", "w") as f: f.write(audit_text)

num_cols = ["Passengers", "Trips", "Average_Delay_Minutes", "On_Time_Percentage",
            "Ticket_Revenue_EUR", "Operating_Cost_EUR", "CO2_Saved_KG",
            "Energy_Consumption_KWh", "Customer_Rating", "Accessibility_Complaints"]
desc = df[num_cols].describe().T[["mean", "std", "min", "50%", "max"]].round(2)
desc.to_csv(OUT + "describe.csv")

# Data-audit "screenshot" (console evidence)
fig = plt.figure(figsize=(8.6, 7.4))
fig.patch.set_facecolor("#1e1e1e")
fig.text(0.02, 0.98, audit_text, family="DejaVu Sans Mono", fontsize=8.6,
         color="#d4d4d4", va="top")
fig.savefig(OUT + "fig0_audit.png", facecolor="#1e1e1e"); plt.close(fig)

# ---------------------------------------------------------------- 3. summary tables
AGG = dict(Records=("Passengers", "size"), Passengers=("Passengers", "sum"),
           Delay=("Average_Delay_Minutes", "mean"), OnTime=("On_Time_Percentage", "mean"),
           Revenue=("Ticket_Revenue_EUR", "sum"), Cost=("Operating_Cost_EUR", "sum"),
           CO2=("CO2_Saved_KG", "sum"), Rating=("Customer_Rating", "mean"),
           Complaints=("Accessibility_Complaints", "sum"))
def summarise(by):
    t = df.groupby(by, observed=True).agg(**AGG)
    t["Share"] = t["Passengers"] / t["Passengers"].sum() * 100
    t["CostRecovery"] = t["Revenue"] / t["Cost"]
    t["Net"] = t["Revenue"] - t["Cost"]
    t["CO2_per_pax"] = t["CO2"] / t["Passengers"]
    t["Compl_per_1k"] = t["Complaints"] / t["Passengers"] * 1000
    return t.sort_values("Passengers", ascending=False)
mode_t, city_t = summarise("Transport_Mode"), summarise("City")
weather_t, route_t, event_t = summarise("Weather"), summarise("Route_Type"), summarise("Event_Day")
for n, t in [("mode", mode_t), ("city", city_t), ("weather", weather_t),
             ("route", route_t), ("event", event_t)]:
    t.round(4).to_csv(OUT + f"tab_{n}.csv")

# ---------------------------------------------------------------- Fig 1: time series
m = df.groupby("Month").agg(Passengers=("Passengers", "sum"),
                            Records=("Passengers", "size"),
                            Delay=("Average_Delay_Minutes", "mean"),
                            OnTime=("On_Time_Percentage", "mean"))
m["Roll3"] = m["Passengers"].rolling(3, center=True).mean()
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6.4), sharex=True,
                             gridspec_kw={"height_ratios": [1.6, 1]})
a1.plot(m.index, m["Passengers"] / 1e6, color=ACCENT, lw=2, marker="o", ms=4, label="Monthly total")
a1.plot(m.index, m["Roll3"] / 1e6, color=INK, lw=1.4, ls="--", label="3-month rolling mean")
a1.set_ylabel("Passengers (millions)")
a1.set_title("Monthly passenger demand and reliability, Jan 2024 - Dec 2025", loc="left")
for d in [pd.Timestamp("2024-03-01"), pd.Timestamp("2025-04-01"), pd.Timestamp("2025-10-01")]:
    v = m.loc[d, "Passengers"] / 1e6
    a1.annotate(f"{d:%b %Y}\n{v:.2f}m", (d, v), textcoords="offset points",
                xytext=(0, 10 if d.month != 10 else -30), ha="center", fontsize=8.5, color=INK)
a1.legend(frameon=False, loc="lower left"); a1.set_ylim(1.2, 2.25)
a2.plot(m.index, m["Delay"], color=ALERT, lw=2, marker="o", ms=4)
a2.axhline(df["Average_Delay_Minutes"].mean(), color=MUTED, ls=":", lw=1.2)
a2.text(m.index[0], df["Average_Delay_Minutes"].mean() + 0.02, "2-year mean 4.35 min", ha="left",
        va="bottom", fontsize=8.5, color=MUTED)
a2.set_ylabel("Mean delay (min)")
a2.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%b\n%Y"))
a2.xaxis.set_major_locator(matplotlib.dates.MonthLocator(bymonth=[1, 4, 7, 10]))
save(fig, "fig1_timeseries.png")

# ---------------------------------------------------------------- Fig 2: city comparison
ct = city_t.sort_values("Passengers")
ccol = df.drop_duplicates("City").set_index("City")["Country"]
fig, ax = plt.subplots(figsize=(10, 5.6))
bars = ax.barh(ct.index, ct["Passengers"] / 1e6, color=[COUNTRY_COL[ccol[c]] for c in ct.index])
for b, (c, r) in zip(bars, ct.iterrows()):
    ax.text(b.get_width() + 0.03, b.get_y() + b.get_height() / 2,
            f"{r.Passengers/1e6:.2f}m  |  {r.Passengers/r.Records:,.0f} per record",
            va="center", fontsize=8.6, color=INK)
ax.axvline(ct["Passengers"].mean() / 1e6, color=MUTED, ls=":", lw=1.2)
ax.text(ct["Passengers"].mean() / 1e6, -0.9, "city mean", ha="center", fontsize=8.5, color=MUTED)
ax.set_xlabel("Total passengers, 2024-2025 (millions)"); ax.set_xlim(0, 6.6); ax.grid(axis="y", visible=False)
ax.set_title("Total passenger demand by city (colour = country)", loc="left")
handles = [plt.Rectangle((0, 0), 1, 1, color=v) for v in COUNTRY_COL.values()]
ax.legend(handles, COUNTRY_COL.keys(), frameon=True, framealpha=1, edgecolor="#ddd", loc="lower right", ncol=1, fontsize=8.8, title="Country")
save(fig, "fig2_city.png")

# ---------------------------------------------------------------- Fig 3: correlation heatmap
labels = {"Passengers": "Passengers", "Trips": "Trips", "Average_Delay_Minutes": "Delay (min)",
          "On_Time_Percentage": "On-time %", "Ticket_Revenue_EUR": "Revenue",
          "Operating_Cost_EUR": "Operating cost", "CO2_Saved_KG": "CO2 saved",
          "Energy_Consumption_KWh": "Energy (kWh)", "Customer_Rating": "Rating",
          "Accessibility_Complaints": "Access. complaints"}
corr = df[num_cols].corr(method="pearson").rename(index=labels, columns=labels)
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(9, 7.4))
sns.heatmap(corr, mask=mask, cmap="RdBu_r", vmin=-1, vmax=1, annot=True, fmt=".2f",
            annot_kws={"size": 8.5}, linewidths=0.6, linecolor="white",
            cbar_kws={"label": "Pearson r", "shrink": 0.75}, ax=ax)
ax.set_title("Correlation matrix of operational, financial and\nsustainability metrics (n = 5,000)", loc="left")
ax.grid(False); plt.xticks(rotation=40, ha="right")
save(fig, "fig3_corr.png")

# ---------------------------------------------------------------- Fig 4: scatter delay vs rating
fig, ax = plt.subplots(figsize=(10, 6))
for mode, col in MODE_COL.items():
    s = df[df["Transport_Mode"] == mode]
    jitter = np.random.default_rng(1).uniform(-0.03, 0.03, len(s))
    ax.scatter(s["Average_Delay_Minutes"], s["Customer_Rating"] + jitter, s=11, alpha=0.45,
               color=col, label=f"{mode} (mean delay {s['Average_Delay_Minutes'].mean():.1f})",
               edgecolors="none")
b1, b0 = np.polyfit(df["Average_Delay_Minutes"], df["Customer_Rating"], 1)
xs = np.linspace(0, 12.5, 50)
ax.plot(xs, b0 + b1 * xs, color=INK, lw=2)
r = df["Average_Delay_Minutes"].corr(df["Customer_Rating"])
ax.text(8.0, 4.72, f"OLS fit: rating = {b0:.2f} {b1:+.3f} x delay\nr = {r:.2f}, R^2 = {r*r:.2f}",
        fontsize=9, color=INK)
ax.set_xlabel("Average delay (minutes)"); ax.set_ylabel("Customer rating (1-5)")
ax.set_title("Each extra minute of delay is associated with lower customer rating", loc="left")
ax.legend(frameon=False, fontsize=8.5, markerscale=2, loc="lower left")
save(fig, "fig4_scatter.png")

# ---------------------------------------------------------------- Fig 5: box plot by weather
order = df.groupby("Weather")["Average_Delay_Minutes"].median().sort_values().index
fig, ax = plt.subplots(figsize=(10, 5.6))
pal = {w: ("#c0392b" if w == "Snow" else "#e8a598" if w == "Rain" else "#9fb3c8") for w in order}
sns.boxplot(data=df, x="Weather", y="Average_Delay_Minutes", order=order, hue="Weather",
            palette=pal, legend=False, width=0.55, fliersize=2.5, linewidth=1.1, ax=ax)
means = df.groupby("Weather")["Average_Delay_Minutes"].mean().reindex(order)
ax.scatter(range(len(order)), means, marker="D", color=INK, zorder=5, s=26, label="Mean")
for i, w in enumerate(order):
    n = (df["Weather"] == w).sum(); ot = weather_t.loc[w, "OnTime"]
    ax.text(i, 13.3, f"n={n:,}\non-time {ot:.0f}%", ha="center", fontsize=8.3, color=MUTED)
ax.set_ylim(-0.5, 14.6); ax.set_xlabel("Weather condition (ordered by median delay)")
ax.set_ylabel("Average delay (minutes)"); ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(0, 0.86))
ax.set_title("Distribution of delay minutes by weather condition", loc="left", pad=14)
save(fig, "fig5_box.png")

# ---------------------------------------------------------------- Fig 6: performance indicators by mode
mt = mode_t.sort_values("Net")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 5.2), gridspec_kw={"wspace": 0.55})
cols = [ALERT if v < 0 else "#2e7d32" for v in mt["Net"]]
a1.barh(mt.index, mt["Net"] / 1e6, color=cols)
a1.axvline(0, color=INK, lw=1)
for i, (mname, rr) in enumerate(mt.iterrows()):
    x = rr.Net / 1e6
    a1.text(max(x, 0) + 0.15, i, f"€{x:+.2f}m\n(recovery {rr.CostRecovery:.2f})",
            va="center", ha="left", fontsize=8.3, color=INK)
a1.set_xlim(-6.2, 7.6); a1.set_xlabel("Net result: revenue - operating cost (€m)")
a1.set_title("(a) Financial performance", loc="left", fontsize=11.5); a1.grid(axis="y", visible=False)
ct2 = mode_t.sort_values("CO2")
a2.barh(ct2.index, ct2["CO2"] / 1000, color=[MODE_COL[mm] for mm in ct2.index])
for i, (mname, rr) in enumerate(ct2.iterrows()):
    a2.text(rr.CO2 / 1000 + 40, i, f"{rr.CO2/1000:,.0f} t  |  {rr.CO2_per_pax*1000:.0f} g/pax",
            va="center", fontsize=8.3)
a2.set_xlim(0, 4600); a2.set_xlabel("Total CO2 saved, 2024-2025 (tonnes)")
a2.set_title("(b) Environmental performance", loc="left", fontsize=11.5); a2.grid(axis="y", visible=False)
fig.suptitle("Financial and environmental performance indicators by transport mode",
             x=0.02, ha="left", fontweight="bold", fontsize=13, y=1.02)
save(fig, "fig6_kpi.png")

# ---------------------------------------------------------------- Fig 7: event-day lollipop
ev = df.groupby("Event_Day", observed=True).agg(
    Passengers=("Passengers", "mean"), Revenue=("Ticket_Revenue_EUR", "mean"),
    Delay=("Average_Delay_Minutes", "mean"), OnTime=("On_Time_Percentage", "mean"),
    Rating=("Customer_Rating", "mean"), Complaints=("Accessibility_Complaints", "mean"))
pct = ((ev.loc["Yes"] / ev.loc["No"]) - 1) * 100
names = {"Passengers": "Passengers per record", "Revenue": "Ticket revenue per record",
         "Delay": "Average delay (min)", "OnTime": "On-time %", "Rating": "Customer rating",
         "Complaints": "Accessibility complaints"}
pct = pct.rename(names).sort_values()
good_up = {"Passengers per record", "Ticket revenue per record", "On-time %", "Customer rating"}
fig, ax = plt.subplots(figsize=(10, 4.8))
for i, (k, v) in enumerate(pct.items()):
    good = (v > 0) == (k in good_up)
    c = "#2e7d32" if good else ALERT
    ax.hlines(i, 0, v, color=c, lw=2.5); ax.plot(v, i, "o", color=c, ms=9)
    ax.text(v + (1.2 if v >= 0 else -1.2), i, f"{v:+.1f}%", va="center",
            ha="left" if v >= 0 else "right", fontsize=9.2, color=INK)
ax.set_yticks(range(len(pct))); ax.set_yticklabels(pct.index)
ax.axvline(0, color=INK, lw=1); ax.set_xlim(-14, 42); ax.grid(axis="y", visible=False)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_xlabel("Event days vs non-event days (% difference in mean; green = favourable, red = adverse)")
ax.set_title("Event days lift demand but erode reliability and satisfaction", loc="left")
save(fig, "fig7_event.png")

# ---------------------------------------------------------------- key stats for the report
mw = df.pivot_table(index="Transport_Mode", columns="Weather", values="Average_Delay_Minutes").round(2)
mw.to_csv(OUT + "tab_mode_weather.csv")
stats = dict(
    rows=len(df), cols=df.shape[1] - 6, first=str(df["Date"].min().date()), last=str(df["Date"].max().date()),
    total_pax=int(df["Passengers"].sum()), slope=round(b1, 3), intercept=round(b0, 2), r_delay_rating=round(r, 2),
    pax_2024=int(df.loc[df.Year == 2024, "Passengers"].sum()), pax_2025=int(df.loc[df.Year == 2025, "Passengers"].sum()),
    recs_2024=int((df.Year == 2024).sum()), recs_2025=int((df.Year == 2025).sum()),
    event_pct=pct.round(1).to_dict(), checks=checks,
    month_corr_records=round(m["Passengers"].corr(m["Records"]), 2),
)
json.dump(stats, open(OUT + "stats.json", "w"), indent=1, default=str)
print(json.dumps(stats, indent=1, default=str))


{
 "rows": 5000,
 "cols": 17,
 "first": "2024-01-01",
 "last": "2025-12-31",
 "total_pax": 40909721,
 "slope": -0.138,
 "intercept": 4.61,
 "r_delay_rating": -0.82,
 "pax_2024": 20974355,
 "pax_2025": 19935366,
 "recs_2024": 2567,
 "recs_2025": 2433,
 "event_pct": {
  "On-time %": -5.2,
  "Customer rating": -3.8,
  "Ticket revenue per record": 25.3,
  "Passengers per record": 26.2,
  "Accessibility complaints": 28.8,
  "Average delay (min)": 31.0
 },
 "checks": {
  "on_time_in_0_100": true,
  "rating_in_1_5": true,
  "negative_values": 0,
  "zero_energy_rows": 759,
  "zero_energy_modes": [
   "Bike Share"
  ],
  "records_per_day_mean": 6.86
 },
 "month_corr_records": 0.13
}


EcoMove Sustainable Mobility Dashboard (Plotly Dash)
Run:  python dashboard.py   ->  http://127.0.0.1:8050
Filters can also be preset through the URL, e.g. ?mode=Bus&weather=Snow,Rain
so that a manager can bookmark and share a filtered view.

In [2]:

!pip install dash
from urllib.parse import parse_qs
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output

# ------------------------------------------------------------------ data
df = pd.read_excel("EcoMoveMobility.xlsx", sheet_name="Dataset")
df["Date"] = pd.to_datetime(df["Date"], unit="D", origin="1899-12-30")
df["Month"] = df["Date"].dt.to_period("M").dt.to_timestamp()
MODE_COL = {"Metro": "#0072B2", "Tram": "#009E73", "Bus": "#D55E00",
            "Park and Ride": "#CC79A7", "Bike Share": "#E69F00", "E-Scooter": "#56B4E9"}
METRICS = {"Passengers": ("Passengers", "sum"),
           "Average delay (min)": ("Average_Delay_Minutes", "mean"),
           "On-time %": ("On_Time_Percentage", "mean"),
           "Net result (EUR)": ("Net", "sum"),
           "CO2 saved (kg)": ("CO2_Saved_KG", "sum"),
           "Customer rating": ("Customer_Rating", "mean"),
           "Complaints per 1,000 pax": ("CPK", "ratio")}
df["Net"] = df["Ticket_Revenue_EUR"] - df["Operating_Cost_EUR"]
FILTERS = [("country", "Country", "Country"), ("city", "City", "City"),
           ("mode", "Transport mode", "Transport_Mode"), ("route", "Route type", "Route_Type"),
           ("weather", "Weather", "Weather"), ("event", "Event day", "Event_Day")]
PLACEHOLDER = {"country": "All countries", "city": "All cities", "mode": "All modes",
               "route": "All route types", "weather": "All weather", "event": "All days"}

def agg(d, by, metric):
    col, how = METRICS[metric]
    g = d.groupby(by)
    if how == "ratio":
        return (g["Accessibility_Complaints"].sum() / g["Passengers"].sum() * 1000).rename(metric)
    return getattr(g[col], how)().rename(metric)

# ------------------------------------------------------------------ layout
CARD = {"background": "#fff", "border": "1px solid #e3e7eb", "borderRadius": "8px",
        "padding": "10px 14px", "flex": "1", "minWidth": "120px"}
app = Dash(__name__, title="EcoMove Dashboard")
app.layout = html.Div(style={"fontFamily": "Segoe UI, Arial", "background": "#f5f7f9",
                             "padding": "14px 18px", "color": "#1f2933"}, children=[
    dcc.Location(id="url"),
    html.H2("EcoMove Sustainable Mobility Dashboard", style={"margin": "0 0 2px"}),
    html.Div("Decision support for route planning, reliability, sustainability, accessibility "
             "and investment, Jan 2024 - Dec 2025", style={"color": "#616e7c", "marginBottom": "10px"}),
    html.Div(style={"display": "flex", "gap": "14px"}, children=[
        # ---- filter panel
        html.Div(style={**CARD, "flex": "0 0 230px"}, children=[
            html.B("Filters"),
            html.Label("Date range", style={"display": "block", "marginTop": "8px", "fontSize": "13px"}),
            dcc.DatePickerRange(id="dates", min_date_allowed=df["Date"].min(),
                                max_date_allowed=df["Date"].max(), start_date=df["Date"].min(),
                                end_date=df["Date"].max(), display_format="MMM YYYY"),
            *[html.Div([html.Label(lbl, style={"fontSize": "13px", "marginTop": "8px", "display": "block"}),
                        dcc.Dropdown(id=key, options=sorted(df[col].unique()), multi=True,
                                     placeholder=PLACEHOLDER[key])]) for key, lbl, col in FILTERS],
            html.Label("Metric for comparison charts", style={"fontSize": "13px", "marginTop": "10px", "display": "block"}),
            dcc.Dropdown(id="metric", options=list(METRICS), value="Average delay (min)", clearable=False),
        ]),
        # ---- main panel
        html.Div(style={"flex": "1", "minWidth": "0"}, children=[
            html.Div(id="kpis", style={"display": "flex", "gap": "10px", "marginBottom": "12px"}),
            html.Div(style={"display": "grid", "gridTemplateColumns": "1fr 1fr", "gap": "12px"}, children=[
                dcc.Graph(id="trend", style={"height": "300px"}),
                dcc.Graph(id="by_mode", style={"height": "300px"}),
                dcc.Graph(id="by_city", style={"height": "330px"}),
                dcc.Graph(id="heat", style={"height": "330px"}),
            ]),
        ]),
    ]),
])

# ------------------------------------------------------------------ URL -> filter presets
@app.callback([Output(k, "value") for k, _, _ in FILTERS], Input("url", "search"))
def preset(search):
    q = parse_qs((search or "").lstrip("?"))
    return [q[k][0].split(",") if k in q else None for k, _, _ in FILTERS]

# ------------------------------------------------------------------ main callback
@app.callback(Output("kpis", "children"), Output("trend", "figure"), Output("by_mode", "figure"),
              Output("by_city", "figure"), Output("heat", "figure"),
              Input("dates", "start_date"), Input("dates", "end_date"),
              *[Input(k, "value") for k, _, _ in FILTERS], Input("metric", "value"))
def update(start, end, *args):
    *sel, metric = args
    d = df[(df["Date"] >= start) & (df["Date"] <= end)]
    for (key, _, col), vals in zip(FILTERS, sel):
        if vals:
            d = d[d[col].isin(vals)]
    if d.empty:
        empty = go.Figure().update_layout(annotations=[dict(text="No data for this selection", showarrow=False)])
        return [html.Div("No records match the filters", style=CARD)], empty, empty, empty, empty

    rev, cost = d["Ticket_Revenue_EUR"].sum(), d["Operating_Cost_EUR"].sum()
    kpi = [("Passengers", f"{d['Passengers'].sum()/1e6:,.2f}m"),
           ("Avg delay", f"{d['Average_Delay_Minutes'].mean():.2f} min"),
           ("On-time", f"{d['On_Time_Percentage'].mean():.1f}%"),
           ("Cost recovery", f"{rev/cost:.2f}"),
           ("CO2 saved", f"{d['CO2_Saved_KG'].sum()/1000:,.0f} t"),
           ("Rating", f"{d['Customer_Rating'].mean():.2f} / 5"),
           ("Complaints /1k pax", f"{d['Accessibility_Complaints'].sum()/d['Passengers'].sum()*1000:.2f}")]
    cards = [html.Div([html.Div(k, style={"fontSize": "12px", "color": "#616e7c"}),
                       html.Div(v, style={"fontSize": "20px", "fontWeight": "600"})], style=CARD) for k, v in kpi]

    base = dict(template="plotly_white", margin=dict(l=50, r=20, t=50, b=40), font=dict(size=11))
    m = d.groupby("Month").agg(Passengers=("Passengers", "sum"), Delay=("Average_Delay_Minutes", "mean")).reset_index()
    trend = go.Figure()
    trend.add_bar(x=m["Month"], y=m["Passengers"], name="Passengers", marker_color="#9fb3c8")
    trend.add_scatter(x=m["Month"], y=m["Delay"], name="Avg delay (min)", yaxis="y2",
                      mode="lines+markers", line=dict(color="#c0392b", width=2))
    trend.update_layout(**base, title="Monthly demand and delay", legend=dict(orientation="h", y=1.12, x=0.45),
                        yaxis=dict(title="Passengers"), yaxis2=dict(title="Delay (min)", overlaying="y", side="right", tickformat=".1f", showgrid=False))

    bm = agg(d, "Transport_Mode", metric).sort_values().reset_index()
    by_mode = px.bar(bm, x=metric, y="Transport_Mode", orientation="h", color="Transport_Mode",
                     color_discrete_map=MODE_COL, title=f"{metric} by transport mode")
    by_mode.update_layout(**base, showlegend=False, yaxis_title=None)

    bc = agg(d, "City", metric).sort_values().reset_index()
    by_city = px.bar(bc, x=metric, y="City", orientation="h", title=f"{metric} by city",
                     color=metric, color_continuous_scale="Blues")
    by_city.update_layout(**base, yaxis_title=None, coloraxis_showscale=False)

    hm = d.pivot_table(index="Transport_Mode", columns="Weather", values="Average_Delay_Minutes", aggfunc="mean")
    heat = px.imshow(hm.round(1), text_auto=True, color_continuous_scale="Reds", aspect="auto",
                     title="Average delay (min): mode x weather")
    heat.update_layout(**base, xaxis_title=None, yaxis_title=None)
    return cards, trend, by_mode, by_city, heat

if __name__ == "__main__":
    app.run(debug=False, host="127.0.0.1", port=8050)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 91.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [dash]


<IPython.core.display.Javascript object>